# MFA Tutorial — Adapted for Llama 3.2 1B + ICL Prompts
---
Original tutorial uses GPT-2 via TransformerLens. This version:
- Uses **Llama 3.2 1B** via HuggingFace + forward hooks for activation extraction
- Uses **ICL prompt dataset** from `data1.py` instead of `supervised.json`
- Runs on **CUDA**

The MFA training/analysis code from the original repo is used unchanged.

## Imports

In [1]:
print()

In [2]:
from __future__ import annotations
import os
import sys
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Import the ICL dataset builder
from data import create_icl_tasks, format_prompt

## Configuration

In [3]:
model_name = "meta-llama/Llama-3.2-1B"  # change if you have a different path/name
layers = [8]  # which layer(s) to extract residual stream activations from
              # Llama 3.2 1B has 16 layers; layer 8 is a reasonable middle layer
data_device = "cuda"
model_device = "cuda"
factorization_mode = "residual"  # extract residual stream activations

## Load Model

In [4]:
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map=model_device,
)
hf_model.eval()
print(f"Model loaded. Hidden dim: {hf_model.config.hidden_size}, Layers: {hf_model.config.num_hidden_layers}")
print(f"Vocab size: {hf_model.config.vocab_size}")

Loading meta-llama/Llama-3.2-1B...


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded. Hidden dim: 2048, Layers: 16
Vocab size: 128256


## Build ICL Dataset

In [5]:
# Create ICL prompts
n_prompts_per_task = 8
n_demos_per_prompt = 3
tasks = create_icl_tasks(n_prompts_per_task, n_demos_per_prompt)

# Format as strings
prompt_strings = [format_prompt(t) for t in tasks]
relation_labels = [t.relation_name for t in tasks]

print(f"Total prompts: {len(prompt_strings)}")
print(f"Relations: {sorted(set(relation_labels))}")
print(f"\nExample prompt:\n{prompt_strings[0]}")

Total prompts: 56
Relations: ['agent_noun', 'capital', 'comparative', 'gender', 'language', 'past_tense', 'plural']

Example prompt:
France -> Paris
Japan -> Tokyo
Italy -> Rome
Egypt ->


## Extract Activations

We hook into the residual stream after each decoder layer and collect activations for **every token position** in each prompt. This replaces the TransformerLens-based `ActivationGenerator`.

In [6]:
def extract_residual_activations(model, tokenizer, prompts, layer_indices, device):
    """
    Extract residual stream activations from specified layers for all token positions.
    
    Returns:
        activations: dict mapping layer_idx -> tensor of shape [total_tokens, hidden_dim]
        all_token_ids: tensor of shape [total_tokens] with the token id at each position
    """
    activations_by_layer = {l: [] for l in layer_indices}
    all_token_ids = []
    
    for prompt in tqdm(prompts, desc="Extracting activations"):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs["input_ids"]  # [1, seq_len]
        
        # Storage for hooks
        hook_outputs = {}
        hooks = []
        
        def make_hook(layer_idx):
            def hook_fn(module, input, output):
                # For LlamaDecoderLayer, output is a tuple; output[0] is the hidden state
                hook_outputs[layer_idx] = output[0].detach()
            return hook_fn
        
        # Register hooks
        for l in layer_indices:
            h = model.model.layers[l].register_forward_hook(make_hook(l))
            hooks.append(h)
        
        # Forward pass
        with torch.no_grad():
            model(**inputs)
        
        # Remove hooks
        for h in hooks:
            h.remove()
        
        # Collect activations for all token positions
        seq_len = input_ids.shape[1]
        for l in layer_indices:
            # hook_outputs[l] shape: [1, seq_len, hidden_dim]
            activations_by_layer[l].append(hook_outputs[l][0].cpu())  # [seq_len, hidden_dim]
        
        # Collect token IDs
        all_token_ids.append(input_ids[0].cpu())  # [seq_len]
    
    # Concatenate across all prompts
    result_acts = {}
    for l in layer_indices:
        result_acts[l] = torch.cat(activations_by_layer[l], dim=0)  # [total_tokens, hidden_dim]
    all_token_ids = torch.cat(all_token_ids, dim=0)  # [total_tokens]
    
    return result_acts, all_token_ids


print("Extracting activations...")
acts_by_layer, all_token_ids = extract_residual_activations(
    hf_model, tokenizer, prompt_strings, layers, model_device
)

for l in layers:
    print(f"Layer {l}: {acts_by_layer[l].shape}")
print(f"Token IDs: {all_token_ids.shape}")

Extracting activations...


Extracting activations: 100%|██████████| 56/56 [00:01<00:00, 54.70it/s]

Layer 8: torch.Size([114688])
Token IDs: torch.Size([896])


## Prepare Data for MFA

The MFA code expects a DataLoader yielding `(activations, token_ids)` batches — same format as the original notebook.

In [7]:
# Use the first (or only) layer
target_layer = layers[0]
X_all = acts_by_layer[target_layer].to(data_device)
tokens = all_token_ids.to(data_device)

print(f"X_all shape: {X_all.shape}")
print(f"tokens shape: {tokens.shape}")

# Create DataLoader (same interface as original notebook)
full_ds = TensorDataset(X_all, tokens)

loader = DataLoader(
    full_ds,
    batch_size=128,
    shuffle=True,
    pin_memory=False,  # already on CUDA
)

token_loader = DataLoader(tokens, batch_size=128)

X_all shape: torch.Size([114688])
tokens shape: torch.Size([896])


## Initialization

The original uses KMeans or random centroids. With a smaller dataset (ICL prompts have far fewer tokens than 250k), adjust `num_centroids` accordingly.

In [8]:
N = X_all.shape[0]
print(f"Total activations: {N}")

# Adjust num_centroids to dataset size
# Rule of thumb: don't exceed N/10 or so
num_centroids = min(500, N // 10)
print(f"Using {num_centroids} centroids")

Total activations: 114688
Using 500 centroids


In [9]:
# Option 1: KMeans initialization (if dataset is large enough)
try:
    from initializations.projected_knn import ReservoirKMeans
    pool_size = round(len(loader.dataset) / 5)
    vocab_size = hf_model.config.vocab_size  # 128256 for Llama 3.2
    knn = ReservoirKMeans(num_centroids, pool_size=pool_size, vocab_size=vocab_size, device=model_device, proj_dim=32)
    centroids = knn.fit(loader)
    print("Initialized with KMeans")
except Exception as e:
    print(f"KMeans init failed ({e}), falling back to random initialization")
    idx = torch.randperm(N, device=X_all.device)[:num_centroids]
    centroids = X_all[idx]
    print(f"Initialized with random centroids: {centroids.shape}")

KMeans init failed (index 23041 is out of bounds for dimension 0 with size 896), falling back to random initialization
Initialized with random centroids: torch.Size([500])


## Training

In [ ]:
from modeling.mfa import MFA
from modeling.train import train_nll

mfa_model = MFA(centroids=centroids, rank=10).to(model_device)
train_nll(mfa_model, loader, epochs=10, lr=1e-3)

## Interpretation

We need a `token_to_str` function. The original uses TransformerLens's `model.to_string()`. We use the HuggingFace tokenizer instead.

In [ ]:
def my_token_to_str(tok_id):
    """Convert token ID to string using HuggingFace tokenizer."""
    if isinstance(tok_id, torch.Tensor):
        tok_id = tok_id.item()
    return tokenizer.decode([tok_id])

In [ ]:
from analysis.subspace_interpretation import get_top_strings_per_concept

results = get_top_strings_per_concept(mfa_model, loader, my_token_to_str, score='likelihood')

In [ ]:
import random

N_RESULTS = 25
N_LINES   = 10
TOP_POOL  = 5000
SEED      = 0

random.seed(SEED)

for i, (r, w) in enumerate(list(results.items())[:N_RESULTS], start=0):
    pool = w[:min(TOP_POOL, len(w))]
    sample = random.sample(pool, k=min(N_LINES, len(pool)))

    print(f"\n[{i}]\n" + "-" * 40)
    for line in sample:
        print("  - " + str(line).replace("\n", "\\n"))

## Visualizing

In [ ]:
import analysis.subspace_visualization as sv

k_to_visualize = min(20, num_centroids - 1)

coords = sv.project_loader_to_subspace(mfa_model, loader, k=k_to_visualize, token_to_str=my_token_to_str)

In [ ]:
sv.plot_subspace_scatter(coords, dims=(0, 8), max_labels=250)

## Steering

**Note:** The steering code in the original repo uses TransformerLens hooks (`MFASteerer` wraps a `HookedTransformer`). To use steering with a HuggingFace model, you would need to adapt `MFASteerer` to use HF forward hooks instead. Below is a sketch — you may need to modify `intervention/mfa_steering.py`.

In [ ]:
# Steering requires adapting MFASteerer for HuggingFace models.
# The core idea is the same: hook into a layer, modify the residual stream
# by interpolating toward a centroid or centroid + z * loadings.
#
# Sketch:
#
# class HFMFASteerer:
#     def __init__(self, hf_model, tokenizer, mfa_model):
#         self.hf_model = hf_model
#         self.tokenizer = tokenizer  
#         self.mfa_model = mfa_model
#
#     def intervene(self, prompt, layers, alpha, k):
#         centroid = self.mfa_model.centroids[k]  # target centroid
#         
#         def hook_fn(module, input, output):
#             h = output[0]  # [1, seq_len, hidden_dim]
#             # Interpolate last token toward centroid
#             h[0, -1, :] = (1 - alpha) * h[0, -1, :] + alpha * centroid
#             return (h,) + output[1:]
#         
#         handle = self.hf_model.model.layers[layers[0]].register_forward_hook(hook_fn)
#         inputs = self.tokenizer(prompt, return_tensors='pt').to(self.hf_model.device)
#         with torch.no_grad():
#             outputs = self.hf_model(**inputs)
#         handle.remove()
#         return outputs.logits

print("Steering section requires adapting MFASteerer for HuggingFace models.")
print("See the sketch above or modify intervention/mfa_steering.py.")